# Lab 27-06 · Networks, Sensors and Compute in a Car

Companion to [27-06 · Automotive E/E Architecture and ADAS Compute](../../chapters/27-06-automotive-ee-architecture.md). Each part asks you to **predict** first, then run the cell. All parameters are the chapter's assumptions.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import automodel as am
MSGS = [(8, 5), (8, 5), (8, 10), (8, 10), (8, 10), (4, 20), (8, 20), (8, 20), (2, 50), (8, 50), (8, 100), (8, 100)]

## Part 1 · Filling the bus

*Predict* the lowest-priority response at 125 kb/s with one more 8-byte, 100 ms message.

In [2]:
m2 = MSGS + [(8, 100)]
print(f'load {100*am.bus_load(m2, 125):.1f}%, response {am.can_response_time_ms(len(m2)-1, m2, 125):.1f} ms')
rates = np.linspace(1000, 125, 120)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(rates, [am.can_response_time_ms(len(MSGS)-1, MSGS, r) for r in rates])
ax.set_xlabel('bit rate (kb/s)'); ax.set_ylabel('lowest-priority response (ms)'); ax.grid(True, alpha=0.3); fig.tight_layout()

load 96.8%, response 41.0 ms


## Part 2 · Camera budget

*Predict* how many 8 MP, 30 fps, 12-bit cameras fit in 40 Gb/s.

In [3]:
cam = am.camera_gbps(8, 30, 12)
print(f'{40/cam:.1f} cameras -> {math.floor(40/cam)} whole')
n = np.arange(1, 17)
fig, ax = plt.subplots(figsize=(6, 3.5))
for mp in (3, 8, 12):
    ax.plot(n, n * am.camera_gbps(mp, 30, 12), label=f'{mp} MP')
ax.axhline(40, color='k', lw=0.8); ax.set_xlabel('cameras'); ax.set_ylabel('input rate (Gb/s)'); ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

13.9 cameras -> 13 whole


## Part 3 · Range in the city

*Predict* the range lost to 500 W at 30 km/h and 140 Wh/km.

In [4]:
print(f'{100*am.range_loss_fraction(500, 140, 30):.1f}% of range')
v = np.linspace(20, 130, 111)
fig, ax = plt.subplots(figsize=(6, 3.5))
for pc in (100, 500):
    ax.plot(v, [100*am.range_loss_fraction(pc, 150, s) for s in v], label=f'{pc} W')
ax.set_xlabel('speed (km/h)'); ax.set_ylabel('range lost (%)'); ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

10.6% of range


## Part 4 · Latency and stopping

*Predict* the stopping distance at 130 km/h with 0.5 s of latency and 8 m/s² braking.

In [5]:
print(f'{am.stopping_distance_m(130, 0.5, 8):.1f} m')
lat = np.linspace(0, 1.0, 101)
fig, ax = plt.subplots(figsize=(6, 3.5))
for s in (50, 100, 130):
    ax.plot(lat, [am.stopping_distance_m(s, t, 8) for t in lat], label=f'{s} km/h')
ax.set_xlabel('latency (s)'); ax.set_ylabel('stopping distance (m)'); ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

99.6 m


## Discussion

Compare your predictions with the [lab README](README.md). Which of these four curves would you bring to a design review for a new central computer, and why?